In [0]:
from pyspark.sql import functions as F

## Product

In [0]:
df_products = spark.table('ecommerce.silver.silver_products')
df_brands = spark.table('ecommerce.silver.silver_brands')
df_category = spark.table('ecommerce.silver.silver_category')


In [0]:
df_products.select('*').limit(5).display()

In [0]:
df_category.select('*').limit(5).display()

In [0]:
df_brands.select('*').limit(5).display()

In [0]:
df_brands.createOrReplaceTempView('v_brands')
df_category.createOrReplaceTempView('v_category')
df_products.createOrReplaceTempView('v_products')


In [0]:
%sql

use ecommerce.silver

In [0]:
%sql
create or replace table gold.gold_products as
select 
    p.product_id,
    p.sku,
    p.category_code,
    coalesce(c.category_name, 'Not Available') as category_name,
    p.brand_code,
    coalesce(b.brand_name, 'Not Available') as brand_name,
    p.color,
    p.size,
    p.material,
    p.weight_grams,
    p.length_cm,
    p.width_cm,
    p.height_cm,
    p.rating_count
from v_products p 
left join v_category c 
    on p.category_code = c.category_code
left join v_brands b 
    on p.brand_code = b.brand_code 
    and p.category_code = b.category_code

## Customer

In [0]:
# India states
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}
# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast", "CA": "West", 
    "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}

# Combine into a master dictionary
country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}  


In [0]:
country_state_map.items()

In [0]:
from pyspark.sql import Row

In [0]:
rows = []
for country,states in country_state_map.items():
    for state_code,region in states.items():
        rows.append(Row(country=country,state=state_code,region=region))
rows[10:]

In [0]:
mapped_df = spark.createDataFrame(rows)
display(mapped_df)

In [0]:
df_customers = spark.table('ecommerce.silver.silver_customers')

In [0]:
df_customers_g = df_customers.alias('c').join(mapped_df.alias('m'),on=['country','state'],how='left')
df_customers_g=df_customers_g.fillna({'region':'Other'})

In [0]:
df_customers_g.select('region').distinct().show()

In [0]:
df_customers_g = df_customers_g.drop('ingested_at','_source_file')

In [0]:
df_customers_g.columns

In [0]:
df_customers_g.write.format('delta').option('mergeSchema',True).mode('overwrite').saveAsTable('ecommerce.gold.gold_customers')

## Calender

In [0]:
df_date = spark.table('ecommerce.silver.silver_date')
df_date.display()

In [0]:
df_calender = df_date.withColumn('date_id',F.date_format(F.col('date'),'yyyyMMdd').cast('int'))
df_calender = df_calender.withColumn('month_name',F.date_format(F.col('date'),'MMMM'))
df_calender = df_calender.withColumn('is_weekend',F.when(F.col('day_name').isin('Satursday','Sunday'),True).otherwise(False))


In [0]:
df_calender.display()

In [0]:
df_calender=df_calender.drop('_source_file','ingested_at')

In [0]:
df_calender.write.format('delta').option('mergeSchema',True).mode('overwrite').saveAsTable('ecommerce.gold.gold_calender')

## Orders

In [0]:
df_orders = spark.table('ecommerce.silver.silver_orders')
df_orders.display()

In [0]:
df_orders=df_orders.fillna({'coupon_code':'None'})

In [0]:
df_orders.limit(3).display()

In [0]:
df_orders.write.format('delta').option('mergeSchema',True).mode('overwrite').saveAsTable('ecommerce.gold.gold_orders')